In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import math

n_val = [10, 100, 1000]
repetitions = 1000

In [ ]:

distributions = {
    "Normal": lambda n: stats.norm.rvs(0, 1, size=n),
    "Cauchy": lambda n: stats.cauchy.rvs(0, 1, size=n),
    "Laplace": lambda n: stats.laplace.rvs(0, scale=1/np.sqrt(2), size=n),
    "Poisson": lambda n: stats.poisson.rvs(5, size=n),
    "Uniform": lambda n: stats.uniform.rvs(-np.sqrt(3), 2*np.sqrt(3), size=n)
}

In [8]:

def calculate_characteristics(x):
    x_sorted = np.sort(x)
    mean = np.mean(x)
    med = np.median(x)
    z_r = (x_sorted[0] + x_sorted[-1]) / 2
    z_q = (np.percentile(x, 25) + np.percentile(x, 75)) / 2
    z_tr = stats.trim_mean(x, 0.1)
    return [mean, med, z_r, z_q, z_tr]

In [ ]:

def generate_results(distributions, n_val, repetitions,
                           format_E="{:.6f}", format_D="{:.6f}", format_interval="[{: .5f}; {: .5f}]", format_hat="{:.1f} ± {:.1f}"):
    all_rows = []
    metric_names = ["x_mean", "med x", "z_R", "z_Q", "z_tr"]
    
    for dist_name, dist_func in distributions.items():
        for n in n_val:
            metrics_runs = np.zeros((repetitions, len(metric_names)))
            for i in range(repetitions):
                sample = dist_func(n)
                metrics_runs[i, :] = calculate_characteristics(sample)
            
            e_z = np.mean(metrics_runs, axis=0)
            d_z = np.var(metrics_runs, axis=0)
            s_z = np.sqrt(d_z)
            
            header_row = [f"{dist_name} n={n}"] + metric_names
            row_e = ["E(z)"] + [format_E.format(v) for v in e_z]
            row_d = ["D(z)"] + [format_D.format(v) for v in d_z]
            row_interval = ["E(z) ± √D(z)"] + [format_interval.format(e - s, e + s) for e, s in zip(e_z, s_z)]
            row_hat = ["Ê(z)"] + [format_hat.format(e, s) for e, s in zip(e_z, s_z)]
            
            all_rows.extend([header_row, row_e, row_d, row_interval, row_hat])
    
    final_df = pd.DataFrame(all_rows)
    return final_df

final_results = generate_results(distributions, n_val, repetitions)

final_results.head(50)


,0,1,2,3,4,5
0,Normal n=10,x_mean,med x,z_R,z_Q,z_tr
1,E(z),0.012315,0.010298,0.005428,0.010788,0.014037
2,D(z),0.100464,0.141029,0.189554,0.112192,0.105355
3,E(z) ± √D(z),[-0.30464; 0.32928],[-0.36524; 0.38584],[-0.42995; 0.44081],[-0.32416; 0.34574],[-0.31055; 0.33862]
4,Ê(z),0.0 ± 0.3,0.0 ± 0.4,0.0 ± 0.4,0.0 ± 0.3,0.0 ± 0.3
5,Normal n=100,x_mean,med x,z_R,z_Q,z_tr
6,E(z),-0.001563,0.000016,0.015290,-0.002767,-0.001627
7,D(z),0.011135,0.017030,0.100772,0.013617,0.011858
8,E(z) ± √D(z),[-0.10709; 0.10396],[-0.13048; 0.13051],[-0.30216; 0.33274],[-0.11946; 0.11393],[-0.11052; 0.10727]
9,Ê(z),-0.0 ± 0.1,0.0 ± 0.1,0.0 ± 0.3,-0.0 ± 0.1,-0.0 ± 0.1
